In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [2]:
dataset = pd.read_csv(
            r"D:\python1\Week-9.1-Machine-Learning-Data-Science-Capstone-SLA-Prediction-Project\2.Data preprocessing\preprocessed_data.csv"
        )

In [3]:
dataset

,Incident_ID,Incident_Type,Priority,Assigned_Department,Location,Status,Resolution_Type,Resolution_Time_Hours,Hour,Day,Month,SLA_Limit,SLA_Breached
0,INC100000,Network Outage,Low,Security Team,Data Center B,Resolved,Reboot,18.0,4,6,3,24,0
1,INC100001,Database Failure,Low,Database Admin,Remote Site 1,Resolved,Reboot,35.0,20,3,1,24,0
2,INC100002,Server Crash,Medium,Database Admin,Remote Site 2,Closed,Patch Applied,72.0,1,2,1,12,1
3,INC100003,Database Failure,Critical,Network Team,Remote Site 1,Resolved,Reboot,36.0,3,5,7,4,1
4,INC100004,Server Crash,Critical,Security Team,Head Office,Resolved,Configuration Fix,28.0,1,4,2,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,INC101195,Application Bug,Critical,Security Team,Remote Site 2,Resolved,Hardware Replacement,47.0,13,4,10,4,1
1196,INC101196,Security Breach,Low,Network Team,Data Center B,Resolved,Reboot,25.0,17,4,1,24,1
1197,INC101197,Network Outage,Medium,IT Support,Data Center B,Resolved,Patch Applied,19.0,5,3,6,12,1
1198,INC101198,Application Bug,Medium,IT Support,Head Office,Closed,Patch Applied,60.0,17,3,9,12,1


In [4]:
dataset.dtypes

Incident_ID                  str
Incident_Type                str
Priority                     str
Assigned_Department          str
Location                     str
Status                       str
Resolution_Type              str
Resolution_Time_Hours    float64
Hour                       int64
Day                        int64
Month                      int64
SLA_Limit                  int64
SLA_Breached               int64
dtype: object

In [5]:
X=dataset.drop("SLA_Breached",axis=1)
Y=dataset["SLA_Breached"]

In [6]:
X.shape

(1200, 12)

In [7]:
# split first (VERY IMPORTANT)
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)


In [8]:
cat_cols = dataset.select_dtypes(include=["string", "object"]).columns

In [9]:
cat_cols

Index(['Incident_ID', 'Incident_Type', 'Priority', 'Assigned_Department',
       'Location', 'Status', 'Resolution_Type'],
      dtype='str')

In [10]:
# categorical columns 
cat_cols=["Incident_ID","Incident_Type","Priority","Assigned_Department","Location","Status","Resolution_Type"]

In [11]:
# one hot encoding 
preprocessor=ColumnTransformer(transformers=[("cat",OneHotEncoder(handle_unknown='ignore',
                sparse_output=False),cat_cols)],remainder="passthrough")
X_encoded=preprocessor.fit_transform(X)
feature_names=preprocessor.get_feature_names_out()
print("Encoded Shape:",X_encoded.shape)

Encoded Shape: (1200, 1231)


In [12]:
# fit only on train
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)


In [13]:
# RFE only on training data
estimator = DecisionTreeClassifier(random_state=42)

rfe = RFE(estimator=estimator, n_features_to_select=20, step=10)
X_train_rfe = rfe.fit_transform(X_train_enc, y_train)
X_test_rfe = rfe.transform(X_test_enc)

In [14]:
# final model
final_model = DecisionTreeClassifier(random_state=42)
final_model.fit(X_train_rfe, y_train)

y_pred = final_model.predict(X_test_rfe)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 1.0


In [15]:
joblib.dump(preprocessor, "preprocessor.pkl")
joblib.dump(rfe, "rfe_selector.pkl")
joblib.dump(final_model, "SLA_prediction_model.pkl")

['SLA_prediction_model.pkl']